Live IDS prototype

This notebook prototypes and tests real-time intrusion detection by completing the following:

1. Capture network traffic
2. Convert captured PCAP files to Flow features (CSV)
3. Load the trained ML model
4. Score the flows
5. Generate alerts

Once completed this notebooke will become a standalone py file

In [1]:
#Comparing columns between the CICIDS2017 and a live sample captured using FlowMeter 

import pandas as pd

cic = pd.read_csv(r"E:\\Project Portfolio\\Dissertation\\Final-Year-IDS\\data\\CICIDS2017\\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
cic.columns = cic.columns.str.strip()

FlowM = pd.read_csv(r"E:\\Project Portfolio\\Dissertation\\Final-Year-IDS\\data\\example.pcap_Flow.csv")
FlowM.columns = FlowM.columns.str.strip()

cic_cols = set(cic.columns)
FlowM_cols = set(FlowM.columns)

print("CIC feature count:", len(cic_cols))
print("Live feature count:", len(FlowM_cols))
print("Shared features:", len(cic_cols & FlowM_cols))

print("\nMissing from live (sample):")
print(list((cic_cols - FlowM_cols))[:20])

print("\nExtra in live (sample):")
print(list((FlowM_cols - cic_cols))[:20])

CIC feature count: 79
Live feature count: 84
Shared features: 28

Missing from live (sample):
['Avg Fwd Segment Size', 'Fwd Packet Length Std', 'Packet Length Std', 'Bwd Packet Length Min', 'Destination Port', 'Flow Packets/s', 'ECE Flag Count', 'Fwd Avg Bulk Rate', 'Subflow Bwd Bytes', 'Fwd Header Length', 'Fwd Avg Packets/Bulk', 'Bwd Packet Length Std', 'Subflow Bwd Packets', 'Bwd Packet Length Max', 'Average Packet Size', 'Bwd Avg Packets/Bulk', 'act_data_pkt_fwd', 'Bwd Avg Bulk Rate', 'Avg Bwd Segment Size', 'Max Packet Length']

Extra in live (sample):
['Flow Byts/s', 'Src IP', 'SYN Flag Cnt', 'Fwd IAT Tot', 'Bwd Pkt Len Std', 'Protocol', 'Pkt Len Max', 'Fwd Pkt Len Std', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Subflow Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Act Data Pkts', 'Fwd Pkt Len Mean', 'Fwd Seg Size Avg', 'Flow Pkts/s', 'Fwd Pkt Len Min', 'Fwd Byts/b Avg', 'Bwd Blk Rate Avg', 'Subflow Bwd Byts']


In [2]:
# Rename map to allow for easier future allignment and implement a function that will normalise features
# For both data types
rename_map = {
    "Pkt Len Mean": "Packet Length Mean",
    "Pkt Len Std": "Packet Length Std",
    "Pkt Len Max": "Packet Length Max",
    "Pkt Len Min": "Packet Length Min",

    "TotLen Fwd Pkts": "Total Length of Fwd Packets",
    "TotLen Bwd Pkts": "Total Length of Bwd Packets",

    "Tot Fwd Pkts": "Total Fwd Packets",
    "Tot Bwd Pkts": "Total Backward Packets",

    "Fwd Pkt Len Mean": "Fwd Packet Length Mean",
    "Fwd Pkt Len Std": "Fwd Packet Length Std",
    "Fwd Pkt Len Max": "Fwd Packet Length Max",
    "Fwd Pkt Len Min": "Fwd Packet Length Min",

    "PSH Flag Cnt": "PSH Flag Count",
    "ACK Flag Cnt": "ACK Flag Count",
    "RST Flag Cnt": "RST Flag Count",
    "URG Flag Cnt": "URG Flag Count",
    "SYN Flag Cnt": "SYN Flag Count",

    "Init Fwd Win Byts": "Init_Win_bytes_forward",
    "Init Bwd Win Byts": "Init_Win_bytes_backward",

    "Subflow Fwd Pkts": "Subflow Fwd Packets",
    "Subflow Bwd Pkts": "Subflow Bwd Packets",

    "Subflow Fwd Byts": "Subflow Fwd Bytes",
    "Subflow Bwd Byts": "Subflow Bwd Bytes",

    "Bwd Seg Size Avg": "Avg Bwd Segment Size",
    "Fwd Seg Size Avg": "Avg Fwd Segment Size",
}

def normalise_live_features(df, train_features):
    df = df.rename(columns=rename_map)
    df = df.reindex(columns=train_features, fill_value=0)

    return df

In [3]:
#Loading the previously saved bundel which includes: model, features, threshold, version and creation date

import joblib

bundle_path = r"E:\Project Portfolio\Dissertation\Final-Year-IDS\models\IDS_RF_v1.0"

bundle = joblib.load(bundle_path)
model = bundle["model"]
train_features = bundle["features"]
threshold = float(bundle["threshold"])

print("Loaded Features:", len(train_features))
print("Threshold:", threshold)

Loaded Features: 78
Threshold: 0.1


In [4]:
#normalising the live csv sample

X_norm = normalise_live_features(FlowM, train_features)

print("Normalised shape:", X_norm.shape)
print("Columns Identical? ", list(X_norm.columns) == list(train_features))

Normalised shape: (12, 78)
Columns Identical?  True


In [5]:
# Count how many features ended up all-zero (means missing/unmatched)
all_zero_cols = (X_norm.sum(axis=0) == 0).sum()
print("All-zero columns:", all_zero_cols, "out of", X_norm.shape[1])

# Check for NaN/inf after normalization
import numpy as np
print("Any NaN:", X_norm.isna().any().any())
print("Any inf:", np.isinf(X_norm.to_numpy()).any())

All-zero columns: 42 out of 78
Any NaN: False
Any inf: False


In [6]:
#testing model predicition

import numpy as np

proba = model.predict_proba(X_norm)[:, 1]
pred = (proba >= threshold).astype(int)

print("Rows:", len(pred))
print("Flagged malicious:", int(pred.sum()))
print("Max proba:", float(np.max(proba)))
print("Mean proba:", float(np.mean(proba)))

Rows: 12
Flagged malicious: 6
Max proba: 0.24705416624108578
Mean proba: 0.12421998767587317


In [7]:
# Function to score the file and return the probability of a malicious packet and the prediction
def score_file(csv_path, model, train_features, threshold):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    X = normalise_live_features(df, train_features)

    proba = model.predict_proba(X)[:,1]
    pred = (proba >= threshold).astype(int)

    results = df.copy()
    results["malicious_prob"] = proba
    results["Prediction"] = pred

    return results

#confirmed safe pcap
sample_file = r"E:\\Project Portfolio\\Dissertation\\Final-Year-IDS\\data\\example.pcap_Flow.csv"

results = score_file(sample_file, model, train_features, threshold)

results.head()

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,malicious_prob,Prediction
0,172.66.148.140-192.168.0.24-443-61421-6,192.168.0.24,61421,172.66.148.140,443,6,03/03/2026 05:07:29 pm,18684,0,2,...,0,0,0,0,0,0,0,No Label,0.079711,0
1,172.66.148.140-192.168.0.24-443-61422-6,192.168.0.24,61422,172.66.148.140,443,6,03/03/2026 05:07:29 pm,19852,0,2,...,0,0,0,0,0,0,0,No Label,0.079599,0
2,172.66.148.140-192.168.0.24-443-61420-6,192.168.0.24,61420,172.66.148.140,443,6,03/03/2026 05:07:29 pm,20534,0,2,...,0,0,0,0,0,0,0,No Label,0.077599,0
3,172.66.148.140-192.168.0.24-443-61419-6,192.168.0.24,61419,172.66.148.140,443,6,03/03/2026 05:07:29 pm,21232,0,2,...,0,0,0,0,0,0,0,No Label,0.077599,0
4,192.168.0.24-63.176.195.25-63938-443-6,192.168.0.24,63938,63.176.195.25,443,6,03/03/2026 05:07:29 pm,34515,1,2,...,0,0,0,0,0,0,0,No Label,0.104187,1


--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
import os, time, subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

# --- Paths ---
root = Path(r"E:\Project Portfolio\Dissertation\Final-Year-IDS")
pcap_dir = root/"live"/"pcaps"
flow_dir = root/"live"/"flows"
alert_dir = root/"live"/"alerts"
alert_csv = alert_dir/"alerts.csv"

#Iterate through the array and make a the new directory based on the path variables
for d in [pcap_dir, flow_dir, alert_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Model Bundel already loaded (Cell 3)
print("Loaded model. Features:", len(train_features), "Threshold:", threshold)

# CICFlowMeter batch file
cicFlowBat = Path(r"C:\Tools\CICFlowMeter-4.0\bin\cfm.bat")
assert cicFlowBat.exists(), f"Cannot find CICFlowMeter bat: {cicFlowBat}"

Loaded model. Features: 78 Threshold: 0.1


In [9]:
result = subprocess.run(["dumpcap", "-D"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

1. \Device\NPF_{F66FFFE5-46F1-4EEA-8319-A5EAEDC78D15} (Local Area Connection* 8)
2. \Device\NPF_{C60C8BEA-136D-4E70-A1AA-DE950E3B5223} (Local Area Connection* 7)
3. \Device\NPF_{4E4E80C3-1B1F-46BC-918A-2465B94FD857} (Local Area Connection* 6)
4. \Device\NPF_{3350E384-C38F-4CD7-BA41-0D2F80A2E852} (Bluetooth Network Connection 3)
5. \Device\NPF_{D0E67351-138D-4A72-BB05-C199700D45FF} (WiFi)
6. \Device\NPF_{3A20BF10-E273-4B83-92A0-F3C12A05A24D} (Local Area Connection* 10)
7. \Device\NPF_{BCCF7766-11DF-4187-9176-4972769148DE} (Local Area Connection* 9)
8. \Device\NPF_Loopback (Adapter for loopback traffic capture)
9. \Device\NPF_{174CE4F1-39CB-4EAC-805B-2EA6E610D709} (Ethernet)




In [62]:
#Select the desired interface (In this case WiFi)
interface = "5"
rotate_time = 10 #pcap duration per file

capture_cmd = [
    "dumpcap",
    "-i", interface,
    "-b", f"duration:{rotate_time}",
    "-w", str(pcap_dir / "capture.pcap")
]

print("Starting capture:", " ".join(capture_cmd))
capture_proc = subprocess.Popen(capture_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print("Capture PID:", capture_proc.pid)

Starting capture: dumpcap -i 5 -b duration:10 -w E:\Project Portfolio\Dissertation\Final-Year-IDS\live\pcaps\capture.pcap
Capture PID: 26524


In [10]:
# Function to convert pcap to csv using CICFlowMeter
def pcap_to_csv(pcap_path: Path) -> Path:
    output = flow_dir
    cmd = ["cmd", "/c",".\\cfm.bat", str(pcap_path), str(output)]
    completed = subprocess.run(
        cmd,
        cwd=cicFlowBat.parent,   # run inside CICFlowMeter/bin
        capture_output=True,
        text=True
    )

    if completed.returncode != 0:
        raise RuntimeError(
            f"CICFlowMeter failed for {pcap_path.name}\n"
            f"stdout:\n{completed.stdout}\n"
            f"stderr:\n{completed.stderr}"
        )

    if not output.exists():
        raise FileNotFoundError(
            f"Expected CSV not created: {output}\n"
            f"stdout:\n{completed.stdout}\n"
            f"stderr:\n{completed.stderr}"
        )

    return output

In [14]:
# Function that will score the csv using the trained IDS model and return a dataframe containing flow metadata for readability
# malicious probability, prediction, severity level and an alert message

def score_csv(flow_csv: Path) -> pd.DataFrame:

    # Load and define metadata columns
    df = pd.read_csv(flow_csv)
    df.columns = df.columns.str.strip()

    meta_headers = ["Flow ID", "Src IP", "Dst IP", "Src Port", "Dst Port", "Protocol", "Timestamp"]
    meta_check = [c for c in meta_headers if c in df.columns]
    metadata = df[meta_check].copy()

    # Prepare model features
    X = normalise_live_features(df, train_features)

    # Ensure features are matched
    if list(X.columns) != list(train_features):
        raise ValueError("Feature mismatch after normalisation")

    # Prediction
    chance = model.predict_proba(X)[:, 1]
    prediction = (chance >= threshold).astype(int)

    # Severity level definement
    def severity(p):
        if p > 0.80:
            return "CRITICAL"
        elif p > 0.50:
            return "HIGH"
        elif p > 0.25:
            return "MEDIUM"
        elif p > threshold:
            return "LOW"
        else:
            return "SAFE"
        
    #Alert Messages
    alerts = []

    for i, row in metadata.iterrows():
        if prediction[i] == 1:
            msg = (
                f"⚠️ Potential malicious packet detected | "
                f"{row.get('Src IP', '?')}:{row.get('Src Port', '?')} -> "
                f"{row.get('Dst IP', '?')}:{row.get('Dst Port', '?')} | "
                f"Protocol: {row.get('Protocol', '?')} | "
                f"Probability: {chance[i]:.3f}"
            )
        else:
            msg = ""
        alerts.append(msg)

    # Output Dataframe
    results = metadata.copy()
    results["malicious_prob"] = chance
    results["prediction"] = prediction
    results["severity"] = [severity(p) for p in chance]
    results["alert"] = alerts

    return results


In [15]:
# Convert pcap to csv file
flow_pcap = Path(r"E:\Project Portfolio\Dissertation\Final-Year-IDS\live\pcaps\capture_00001_20260305165920.pcap")
output_folder = pcap_to_csv(flow_pcap)
print("CICFlowMeter output folder:", output_folder)

#Find the latest CSV file created
csv_file = flow_dir / f"{flow_pcap.stem}.pcap_Flow.csv"
fallback_csv = sorted(flow_dir.glob(f"{flow_pcap.stem}*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)

print("Using CSV:", csv_file)

# Score the csv file using the IDS model
results = score_csv(csv_file)

print(f"Flows analysed: {len(results)}")
print(f"Alerts generated: {int(results['prediction'].sum())}")
print(f"Max probability: {float(results['malicious_prob'].max())}")

alerts = results[results["prediction"] == 1].sort_values("malicious_prob", ascending=False)

display_cols = [c for c in ["Timestamp","Src IP","Src Port","Dst IP","Dst Port","Protocol",
                            "malicious_prob","severity","alert"] if c in results.columns]
display(alerts[display_cols].head(15))

CICFlowMeter output folder: E:\Project Portfolio\Dissertation\Final-Year-IDS\live\flows
Using CSV: E:\Project Portfolio\Dissertation\Final-Year-IDS\live\flows\capture_00001_20260305165920.pcap_Flow.csv
Flows analysed: 20
Alerts generated: 15
Max probability: 0.24768186350269056


,Timestamp,Src IP,Src Port,Dst IP,Dst Port,Protocol,malicious_prob,severity,alert
5,05/03/2026 04:59:21 pm,192.168.0.24,54915,192.168.0.255,54915,17,0.247682,LOW,⚠️ Potential malicious packet detected | 192.1...
1,05/03/2026 04:59:23 pm,192.168.0.24,50766,52.178.17.3,443,6,0.234374,LOW,⚠️ Potential malicious packet detected | 192.1...
0,05/03/2026 04:59:22 pm,192.168.0.24,50766,52.178.17.3,443,6,0.188559,LOW,⚠️ Potential malicious packet detected | 192.1...
2,05/03/2026 04:59:24 pm,192.168.0.24,50772,104.18.41.168,8443,6,0.187783,LOW,⚠️ Potential malicious packet detected | 192.1...
18,05/03/2026 04:59:29 pm,142.250.129.101,443,192.168.0.24,56513,6,0.186577,LOW,⚠️ Potential malicious packet detected | 142.2...
12,05/03/2026 04:59:23 pm,192.168.0.24,49846,142.250.140.94,443,6,0.145748,LOW,⚠️ Potential malicious packet detected | 192.1...
15,05/03/2026 04:59:22 pm,172.64.148.235,443,192.168.0.24,57194,6,0.140870,LOW,⚠️ Potential malicious packet detected | 172.6...
16,05/03/2026 04:59:22 pm,192.168.0.24,64264,20.42.65.89,443,6,0.139944,LOW,⚠️ Potential malicious packet detected | 192.1...
17,05/03/2026 04:59:29 pm,142.250.151.95,443,192.168.0.24,63996,6,0.138693,LOW,⚠️ Potential malicious packet detected | 142.2...
6,05/03/2026 04:59:24 pm,104.18.41.168,8443,192.168.0.24,50772,6,0.123548,LOW,⚠️ Potential malicious packet detected | 104.1...


In [17]:
# -----------------------------
# Configuration
# -----------------------------

interface = "5"          # change if needed
rotate_time = 10         # seconds per capture file
poll_time = 1.0          # loop interval
min_pcap_age = 5.0       # seconds before processing a pcap
top_alerts = 5

alert_log = alert_dir / "alerts_log.csv"

pcap_dir.mkdir(parents=True, exist_ok=True)
flow_dir.mkdir(parents=True, exist_ok=True)
alert_dir.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Helper functions
# -----------------------------

def pcap_old_enough(pcap_path: Path, age: float):
    return (time.time() - pcap_path.stat().st_mtime) >= age


def find_flow_csv(pcap_path: Path):

    matches = sorted(
        flow_dir.glob(f"{pcap_path.stem}*.csv"),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )

    if not matches:
        raise FileNotFoundError(f"No flow CSV found for {pcap_path.stem}")

    return matches[0]


def log_alert(row):

    df = pd.DataFrame([row])

    if not alert_log.exists():
        df.to_csv(alert_log, index=False)
    else:
        df.to_csv(alert_log, mode="a", header=False, index=False)


# -----------------------------
# Start packet capture
# -----------------------------

capture_cmd = [
    "dumpcap",
    "-i", interface,
    "-b", f"duration:{rotate_time}",
    "-w", str(pcap_dir / "capture.pcap")
]

print("Starting capture:")
print(" ".join(capture_cmd))

capture_proc = subprocess.Popen(
    capture_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print("Capture PID:", capture_proc.pid)
print("Writing PCAP files to:", pcap_dir)


# -----------------------------
# Live IDS loop
# -----------------------------

processed_pcaps = set()

print("\nLive IDS running...")
print("Interrupt the cell to stop.\n")

try:
    while True:
        pcaps = sorted(pcap_dir.glob("*.pcap"), key=lambda p: p.stat().st_mtime)
        for pcap_file in pcaps:

            if pcap_file.name in processed_pcaps:
                continue

            if not pcap_old_enough(pcap_file, min_pcap_age):
                continue

            print(f"\n[PCAP] {pcap_file.name}")

            # Convert PCAP → Flow CSV
            try:
                pcap_to_csv(pcap_file)
                flow_csv = find_flow_csv(pcap_file)
                print("[FLOW]", flow_csv.name)

            except Exception as e:
                print("[FLOW ERROR]", e)
                processed_pcaps.add(pcap_file.name)
                continue

            # Score flows
            try:
                results = score_csv(flow_csv)
                flow_count = len(results)
                alert_count = int(results["prediction"].sum())
                max_prob = float(results["malicious_prob"].max())
                print(f"[SCORE] flows={flow_count} alerts={alert_count} max_prob={max_prob:.3f}")
                alerts = results[results["prediction"] == 1].sort_values(
                    "malicious_prob",
                    ascending=False
                )

                if alert_count > 0:
                    print("[ALERTS]")
                    for alert in alerts["alert"].head(top_alerts):
                        print(alert)
                    top = alerts.iloc[0]

                else:
                    top = None

                log_alert({
                    "time": pd.Timestamp.utcnow().isoformat(),
                    "pcap": pcap_file.name,
                    "flows": flow_count,
                    "alerts": alert_count,
                    "max_prob": max_prob,
                    "src_ip": top.get("Src IP") if top is not None else "",
                    "dst_ip": top.get("Dst IP") if top is not None else "",
                    "severity": top.get("severity") if top is not None else ""
                })

            except Exception as e:
                print("[SCORE ERROR]", e)

            processed_pcaps.add(pcap_file.name)

        time.sleep(poll_time)

except KeyboardInterrupt:
    print("\nStopping IDS...")

finally:
    if capture_proc.poll() is None:
        capture_proc.terminate()

        try:
            capture_proc.wait(timeout=3)
        except subprocess.TimeoutExpired:
            capture_proc.kill()

    print("Capture stopped.")

Starting capture:
dumpcap -i 5 -b duration:10 -w E:\Project Portfolio\Dissertation\Final-Year-IDS\live\pcaps\capture.pcap
Capture PID: 20952
Writing PCAP files to: E:\Project Portfolio\Dissertation\Final-Year-IDS\live\pcaps

Live IDS running...
Interrupt the cell to stop.


[PCAP] capture_00001_20260305165920.pcap
[FLOW] capture_00001_20260305165920.pcap_Flow.csv
[SCORE] flows=20 alerts=15 max_prob=0.248
[ALERTS]
⚠️ Potential malicious packet detected | 192.168.0.24:54915 -> 192.168.0.255:54915 | Protocol: 17 | Probability: 0.248
⚠️ Potential malicious packet detected | 192.168.0.24:50766 -> 52.178.17.3:443 | Protocol: 6 | Probability: 0.234
⚠️ Potential malicious packet detected | 192.168.0.24:50766 -> 52.178.17.3:443 | Protocol: 6 | Probability: 0.189
⚠️ Potential malicious packet detected | 192.168.0.24:50772 -> 104.18.41.168:8443 | Protocol: 6 | Probability: 0.188
⚠️ Potential malicious packet detected | 142.250.129.101:443 -> 192.168.0.24:56513 | Protocol: 6 | Probability: 0.187



C:\Users\Nova_\AppData\Local\Temp\ipykernel_25424\143914341.py:130: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "time": pd.Timestamp.utcnow().isoformat(),


[FLOW] capture_00002_20260305165931.pcap_Flow.csv
[SCORE] flows=7 alerts=7 max_prob=0.242
[ALERTS]
⚠️ Potential malicious packet detected | 192.168.0.24:54915 -> 192.168.0.255:54915 | Protocol: 17 | Probability: 0.242
⚠️ Potential malicious packet detected | 142.251.30.95:443 -> 192.168.0.24:58737 | Protocol: 6 | Probability: 0.185
⚠️ Potential malicious packet detected | 3.75.138.119:2099 -> 192.168.0.24:50393 | Protocol: 6 | Probability: 0.119
⚠️ Potential malicious packet detected | 192.168.0.24:61534 -> 98.95.120.71:443 | Protocol: 6 | Probability: 0.119
⚠️ Potential malicious packet detected | 192.168.0.24:61366 -> 192.178.223.188:5228 | Protocol: 6 | Probability: 0.109


C:\Users\Nova_\AppData\Local\Temp\ipykernel_25424\143914341.py:130: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "time": pd.Timestamp.utcnow().isoformat(),



[PCAP] capture_00001_20260305233907.pcap
[FLOW] capture_00001_20260305233907.pcap_Flow.csv
[SCORE] flows=63 alerts=54 max_prob=0.248
[ALERTS]
⚠️ Potential malicious packet detected | 192.168.0.24:54915 -> 192.168.0.255:54915 | Protocol: 17 | Probability: 0.248
⚠️ Potential malicious packet detected | 192.168.0.24:63658 -> 20.26.156.210:443 | Protocol: 6 | Probability: 0.240
⚠️ Potential malicious packet detected | 192.168.0.24:58265 -> 104.21.66.173:443 | Protocol: 17 | Probability: 0.229
⚠️ Potential malicious packet detected | 192.168.0.24:64812 -> 168.235.193.71:443 | Protocol: 6 | Probability: 0.223
⚠️ Potential malicious packet detected | 192.168.0.24:63256 -> 194.168.4.100:53 | Protocol: 17 | Probability: 0.217


...


[PCAP] capture_00007_20260305234008.pcap
[FLOW] capture_00007_20260305234008.pcap_Flow.csv
[SCORE] flows=24 alerts=21 max_prob=0.248
[ALERTS]
⚠️ Potential malicious packet detected | 192.168.0.24:54915 -> 192.168.0.255:54915 | Protocol: 17 | Probability: 0.248
⚠️ Potential malicious packet detected | 192.168.0.24:52215 -> 104.21.66.173:443 | Protocol: 17 | Probability: 0.223
⚠️ Potential malicious packet detected | 142.250.151.113:443 -> 192.168.0.24:51580 | Protocol: 6 | Probability: 0.210
⚠️ Potential malicious packet detected | 192.178.223.95:443 -> 192.168.0.24:65458 | Protocol: 6 | Probability: 0.207
⚠️ Potential malicious packet detected | 192.168.0.24:50795 -> 20.189.173.11:443 | Protocol: 6 | Probability: 0.195


C:\Users\Nova_\AppData\Local\Temp\ipykernel_25424\143914341.py:130: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "time": pd.Timestamp.utcnow().isoformat(),



[PCAP] capture_00008_20260305234018.pcap
[FLOW] capture_00008_20260305234018.pcap_Flow.csv
[SCORE] flows=11 alerts=9 max_prob=0.248
[ALERTS]
⚠️ Potential malicious packet detected | 192.168.0.24:54915 -> 192.168.0.255:54915 | Protocol: 17 | Probability: 0.248
⚠️ Potential malicious packet detected | 192.168.0.91:5353 -> 224.0.0.251:5353 | Protocol: 17 | Probability: 0.199
⚠️ Potential malicious packet detected | 100.52.55.209:443 -> 192.168.0.24:52005 | Protocol: 6 | Probability: 0.189
⚠️ Potential malicious packet detected | 192.168.0.24:50795 -> 20.189.173.11:443 | Protocol: 6 | Probability: 0.163
⚠️ Potential malicious packet detected | 192.168.0.24:49322 -> 142.250.151.101:443 | Protocol: 6 | Probability: 0.160

[PCAP] capture_00009_20260305234028.pcap


C:\Users\Nova_\AppData\Local\Temp\ipykernel_25424\143914341.py:130: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "time": pd.Timestamp.utcnow().isoformat(),


[FLOW] capture_00009_20260305234028.pcap_Flow.csv
[SCORE ERROR] Found array with 0 sample(s) (shape=(0, 78)) while a minimum of 1 is required by RandomForestClassifier.

Stopping IDS...
Capture stopped.
